# The WSP Heuristic

Here we calculate the WSPD for a given problem and then check whether the number of edges crossing the each pair of clusters is the correct number given the number of exits out of the cluster as per the proved theorems. 

In [ ]:
from pathlib import Path

import tsplib95
import numpy as np

from wsp import ds, tsp

import sys

TREE_TYPE = ds.PKPRQuadTree
SIZE_THRESHOLD = 250
TSP_LIB_PATH = Path("TSPLIB")
S_FACTOR = 2.01
ax = np.array([None, None])
sys.setrecursionlimit(500_000)

In [2]:
all_problems : dict[str, tsp.TravellingSalesmanProblem[TREE_TYPE]] = {}

for file in sorted(TSP_LIB_PATH.iterdir()): # Loop through every tsp
    if file.suffix != ".tsp" or not file.is_file():
        continue
    problem = tsplib95.load(f"{file.absolute()}")
    if problem.edge_weight_type != "EUC_2D": # Skip non-Euclidean TSPs
        continue
    if problem.dimension > SIZE_THRESHOLD: # Skip large TSPs
        continue

    points = [ds.Point(*problem.node_coords[i]) for i in problem.get_nodes()]
    ts_problem = tsp.TravellingSalesmanProblem[TREE_TYPE](TREE_TYPE, points, ax, s=S_FACTOR) 
    
    all_problems[problem.name] = ts_problem
    print(f"Added {problem.name}")

print("Found", len(all_problems), "euclidean TSPs")

Added berlin52
Added eil51
Added eil76
Added kroA100
Added kroB100
Added kroC100
Added kroD100
Added kroE100
Added pr76
Added rat99
Added rd100
Added st70
Found 12 euclidean TSPs


In [ ]:
for name, problem in all_problems.items():
    tour, _, _ = problem.nnn_path
    badCount = 0
    for (A, B), _ in problem.wspd:
        pointsA = set(A.covered_points)
        pointsB = set(B.covered_points)
        if not problem.wsp_heuristic_good(tour, pointsA, pointsB):
            badCount += 1
    if badCount > 0:
        print(f"Problem {name} has {badCount} bad pairs")